# Sales Performance Analysis Project

## The Goal:
To discover some additional insights besides data visualization on sales_performance_dashboard.pbix

## The Proposal:
Use some SQL fundamental techniques:
- Retrieve & Filter Data (SELECT, FROM, WHERE, ORDER BY, DISTINCT, LIMIT)
- Summarize Data (COUNT(), SUM(), AVG(), MIN()/MAX(), GROUP BY, HAVING)
- Combine Tables (INNER JOIN, LEFT JOIN, RIGHT JOIN)
- Advanced SQL for Analysts (CASE WHEN, CTE, WINDOW FUNCTIONS, ROW_NUMBER(), LAG())

## Dataset Overview:

This project uses three related tables extracted from **Plant_DTS.xlsx**. Together, these tables form a relational database that supports customer, product, and sales performance analysis using SQL within Python.



### 1. Accounts

#### Description

The `Accounts` table contains customer information. Each row represents a unique customer account and includes descriptive attributes such as customer name, country, market, and sales channel. This table provides customer and geographic context for sales analysis.

**Primary Key**

- `Account_id`



### 2. Plant_FACT

#### Description

The `Plant_FACT` table is the central **fact table** that stores individual sales transactions. Each record contains key business metrics, including sales revenue, cost of goods sold (COGS), quantity sold, and transaction date. It links customers and products through foreign keys, making it the primary table for business analysis.

**Foreign Keys**

- `Account_id` → `Accounts`
- `Product_id` → `Plant_Hierarchy`

**Business Measures**

- `Sales_USD`
- `COGS_USD`
- `Quantity`



### 3. Plant_Hierarchy

#### Description

The `Plant_Hierarchy` table serves as the **product dimension table**, containing product information and hierarchical classifications. It organizes products into categories such as product family and product group, enabling sales analysis at different levels of the product hierarchy.

**Primary Key**

- `Product_id`

## Database Relationship:

The three tables form a simple **star schema**, where `Plant_FACT` acts as the central fact table connected to two dimension tables.

```text
                Accounts
          (Customer Dimension)
              Account_id
                  │
                  │
                  │
Plant_Hierarchy ─── Plant_FACT
(Product Dimension)  (Fact Table)
    Product_id         Product_id
                       Account_id

### Table Relationships:

| Table | Key | Relationship |
|--------|-----|--------------|
| `Accounts` | `Account_id` | One customer can have multiple sales transactions. |
| `Plant_FACT` | `Account_id`, `Product_id` | Stores transactional sales data and links customers to products. |
| `Plant_Hierarchy` | `Product_id` | One product can appear in multiple sales transactions. |

## Project Workflow:

1. Import the three CSV files into Python using **Pandas**.
2. Create a SQLite database and load each dataset as a separate table.
3. Execute SQL queries to analyze customer, product, and sales performance.
4. Visualize the query results using **Power BI**.

This relational database structure enables comprehensive business analysis by combining transactional sales data with customer and product information through SQL joins.

## Connect datasets into sqlite3 on Python

In [1]:
import pandas as pd
import sqlite3

In [2]:
accounts = pd.read_csv("Accounts.csv")
plant_fact = pd.read_csv("Plant_FACT.csv")
plant_hierarchy = pd.read_csv("Plant_Hierarchy.csv")

In [3]:
conn = sqlite3.connect("plant_database.db")

In [4]:
accounts.to_sql(
    "Accounts",
    conn,
    if_exists="replace",
    index=False
)

plant_fact.to_sql(
    "Plant_FACT",
    conn,
    if_exists="replace",
    index=False
)

plant_hierarchy.to_sql(
    "Plant_Hierarchy",
    conn,
    if_exists="replace",
    index=False
)

1000

## Verify the tables

In [5]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

,name
0,Accounts
1,Plant_FACT
2,Plant_Hierarchy


## View the columns in each table

In [10]:
pd.read_sql("PRAGMA table_info(Accounts);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,country_code,TEXT,0,None,0
1,1,Account,TEXT,0,None,0
2,2,Master_id,INTEGER,0,None,0
3,3,Account_id,TEXT,0,None,0
4,4,latitude2,REAL,0,None,0
5,5,longitude,REAL,0,None,0
6,6,country2,TEXT,0,None,0
7,7,Postal_code,TEXT,0,None,0
8,8,street_name,TEXT,0,None,0
9,9,Street_number,INTEGER,0,None,0


In [11]:
pd.read_sql("PRAGMA table_info(Plant_FACT);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Product_id,INTEGER,0,None,0
1,1,Sales_USD,REAL,0,None,0
2,2,quantity,REAL,0,None,0
3,3,Price_USD,REAL,0,None,0
4,4,COGS_USD,REAL,0,None,0
5,5,Date_Time,TEXT,0,None,0
6,6,Account_id,TEXT,0,None,0


In [12]:
pd.read_sql("PRAGMA table_info(Plant_Hierarchy);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Product_Family,TEXT,0,None,0
1,1,Product_Family_Id,INTEGER,0,None,0
2,2,Product_Group,TEXT,0,None,0
3,3,Product_Group_id,INTEGER,0,None,0
4,4,Product_Name,TEXT,0,None,0
5,5,Product_Name_id,INTEGER,0,None,0
6,6,Product_Size,TEXT,0,None,0
7,7,Produt_Type,TEXT,0,None,0


## Write SQL query to find insights

### Top 10 most profitable products

In [41]:
top_10_profitable = """
SELECT pf.Product_id
     , ph.Product_Name
     , ph.Product_Family
     , ROUND(SUM((Sales_USD - COGS_USD)*quantity)/1e6, 2) AS profit_millions
FROM plant_fact as pf
JOIN plant_hierarchy as ph
ON pf.Product_id = ph.Product_Name_id
GROUP BY pf.Product_id
       , ph.Product_Name
       , ph.Product_Family
ORDER BY profit_millions DESC
LIMIT 10
"""
insight1 = pd.read_sql(top_10_profitable, conn)
insight1

,Product_id,Product_Name,Product_Family,profit_millions
0,2600,Sanicula crassicaulis Poepp. ex DC. var. tripa...,Pyrenulaceae,25.00
1,2859,Nothoscordum texanum M.E. Jones,Malvaceae,24.99
2,2158,Ramalina paludosa B. Moore,Anacardiaceae,24.39
3,2901,Clematis viorna L.,Myrtaceae,23.31
4,2511,Luffa operculata (L.) Cogn.,Poaceae,23.20
5,2913,Phoenicaulis cheiranthoides Nutt.,Myrtaceae,23.02
6,2700,Lecanora cinereofusca H. Magn.,Scrophulariaceae,21.79
7,2940,Verbena urticifolia L. var. leiocarpa L.M. Per...,Asteraceae,21.79
8,2058,Malpighia coccigera L.,Brassicaceae,21.67
9,2561,Typha angustifolia L.,Fabaceae,21.67


### Revenue and Profit by Country

In [28]:
revenue_profit_country = """
SELECT a.country_code
	, ROUND(SUM(pf.Sales_USD * pf.quantity)/1e6,2) AS revenue_millions
	, ROUND(SUM((pf.Sales_USD - pf.COGS_USD)*pf.quantity)/1e6,2) AS profit_millions
FROM plant_fact AS pf
JOIN accounts AS a
ON pf.Account_id = a.Account_id
GROUP BY a.country_code
"""
insight2 = pd.read_sql(revenue_profit_country, conn)
insight2

,country_code,revenue_millions,profit_millions
0,AR,209.39,83.68
1,AS,57.63,17.26
2,AU,16.79,4.46
3,BE,9.40,4.93
4,BG,103.17,50.82
5,BR,1083.12,435.24
6,CA,439.79,164.34
7,CF,50.36,9.46
8,CH,33.19,15.53
9,CI,122.13,47.08


### Product Margin by Product

In [42]:
margin_product = """
SELECT pf.Product_id
    , ph.Product_Name
    , ph.Product_Family
    , ROUND(SUM(Sales_USD*quantity)/1e6, 2) AS revenue_millions
    , ROUND(SUM((Sales_USD - COGS_USD)*quantity)/1e6, 2) AS profit_millios
    , ROUND(ROUND(SUM(Sales_USD*quantity)/1e6, 2)/ROUND(SUM((Sales_USD - COGS_USD)*quantity)/1e6, 2),2)*100 AS profit_margin_pct
FROM plant_fact as pf
JOIN plant_hierarchy as ph
ON pf.Product_id = ph.Product_Name_id
GROUP BY pf.Product_id
       , ph.Product_Name
       , ph.Product_Family
"""
insight3 = pd.read_sql(margin_product, conn)
insight3

,Product_id,Product_Name,Product_Family,revenue_millions,profit_millios,profit_margin_pct
0,2000,Chamaesyce celastroides (Boiss.) Croizat & O. ...,Cucurbitaceae,21.35,10.15,210.0
1,2002,Iris √ónelsonii Randolph,Scrophulariaceae,9.38,3.46,271.0
2,2003,Acanthus L.,Euphorbiaceae,12.94,7.05,184.0
3,2004,Aplectrum hyemale (Muhl. ex Willd.) Torr.,Amaranthaceae,3.53,1.09,324.0
4,2005,Parkia bicolor A. Chev.,Asteraceae,10.94,4.46,245.0
...,...,...,...,...,...,...
910,26761,Hieracium glomeratum Froel. Xx,Bryaceae,10.76,7.04,153.0
911,27061,Rinodina milvina (Wahlenb.) Th. Fr. South Amer...,Rosaceae,18.27,5.38,340.0
912,28211,Anulocaulis leiosolenus (Torr.) Standl. var. h...,Brassicaceae,10.88,3.18,342.0
913,29161,Oonopsis wardii (A. Gray) Greene g23,Lauraceae,14.46,7.05,205.0


### Top 3 Products in each Country

In [43]:
top3_products_country = """
WITH top_products AS(
    SELECT a.country_code
           , pf.Product_id
           , ph.Product_Name
           , ph.Product_Family
           ,ROW_NUMBER() OVER(PARTITION BY a.country_code ORDER BY SUM(Sales_USD) DESC) AS product_rank
FROM plant_fact AS pf
JOIN accounts AS a
ON pf.Account_id = a.Account_id
JOIN plant_hierarchy as ph
ON pf.Product_id = ph.Product_Name_id
GROUP BY a.country_code
        , pf.Product_id
        , ph.Product_Name
        , ph.Product_Family
)

SELECT * FROM top_products
WHERE product_rank <= 3
"""
insight4 = pd.read_sql(top3_products_country, conn)
insight4

,country_code,Product_id,Product_Name,Product_Family,product_rank
0,AR,2650,Porpidia K√∂rb.,Saxifragaceae,1
1,AR,2788,Mitella caulescens Nutt.,Apiaceae,2
2,AR,2639,Lechea sessiliflora Raf.,Asteraceae,3
3,AS,2849,Epilobium brachycarpum C. Presl,Caryophyllaceae,1
4,AS,2927,Eurybia conspicua (Lindl.) G.L. Nesom,Zingiberaceae,2
...,...,...,...,...,...
138,VN,2008,Agave L.,Rosaceae,2
139,VN,2484,Lecidea plebeja Nyl.,Lamiaceae,3
140,ZA,2013,Plantago princeps Cham. & Schltdl.,Agavaceae,1
141,ZA,2868,Mitella trifida Graham var. trifida,Agavaceae,2


### High-Revenue, Low-Profit Products

In [44]:
highrevenue_lowprofit = """
WITH ProductPerformance AS(
    SELECT pf.Product_id
        , ph.Product_Name
        , ph.Product_Family
        , SUM(pf.Sales_USD) AS total_revenue
        , SUM(pf.Sales_USD - pf.COGS_USD) AS total_profit
        , (SUM(pf.Sales_USD - pf.COGS_USD)/SUM(pf.Sales_USD))*100 AS profit_margin_pct
    FROM plant_fact as pf
    JOIN plant_hierarchy as ph
    ON pf.Product_id = ph.Product_Name_id
    GROUP BY pf.Product_id
           , ph.Product_Name
           , ph.Product_Family
),

Benchmarks AS (
    SELECT AVG(total_revenue) AS avg_revenue
        , AVG(profit_margin_pct) AS avg_margin_pct
    FROM ProductPerformance
)

SELECT p.Product_id
    , p.Product_Name
    , p.Product_Family
    , ROUND(p.total_revenue, 2) AS total_revenue
    , ROUND(p.total_profit, 2) AS total_profit
    , ROUND(p.profit_margin_pct, 2) AS profit_margin_pct
FROM ProductPerformance p
CROSS JOIN Benchmarks b
WHERE p.total_revenue > b.avg_revenue     
AND p.profit_margin_pct < b.avg_margin_pct
"""
insight5 = pd.read_sql(highrevenue_lowprofit, conn)
insight5

,Product_id,Product_Name,Product_Family,total_revenue,total_profit,profit_margin_pct
0,2010,Scutellaria alabamensis Alexander,Fabaceae,59753.16,23269.20,38.94
1,2017,Boykinia aconitifolia Nutt.,Verbenaceae,49901.03,17057.57,34.18
2,2027,Aniba Aubl.,Cyperaceae,34758.99,10543.39,30.33
3,2028,Sisyrinchium rosulatum E.P. Bicknell,Ophioglossaceae,83366.99,23396.68,28.06
4,2043,Ivesia arizonica (Eastw. ex J.T. Howell) Ertte...,Asteraceae,59404.59,20677.45,34.81
...,...,...,...,...,...,...
196,2969,Ipomoea wrightii A. Gray,Asteraceae,36660.86,9679.52,26.40
197,2980,Eriastrum luteum (Benth.) H. Mason,Campanulaceae,63665.77,16825.82,26.43
198,2985,Luculia intermedia Hutch.,Fabaceae,41741.59,11362.20,27.22
199,21251,Quercus macrocarpa Michx. - 12m,Fagaceae,62146.23,21523.22,34.63


### Pareto Analysis (80-20 Contribution)

In [46]:
pareto_analysis = """
WITH ProductRevenue AS(
    SELECT pf.Product_id
        , ph.Product_Name
        , ph.Product_Family
        , SUM(Sales_USD) AS product_revenue
    FROM plant_fact as pf
    JOIN plant_hierarchy as ph
    ON pf.Product_id = ph.Product_Name_id
    GROUP BY pf.Product_id
           , ph.Product_Name
           , ph.Product_Family
),

CumulativeRevenue AS (
    SELECT Product_id
        , Product_Name
        , Product_Family
        , product_revenue
        , SUM(product_revenue) OVER (ORDER BY product_revenue DESC) AS running_total_revenue
        , SUM(product_revenue) OVER () AS total_company_revenue
    FROM ProductRevenue
),

ParetoCalculation AS (
    SELECT Product_id
        , Product_Name
        , Product_Family
        , product_revenue
        , running_total_revenue
        , total_company_revenue
        , ROUND((running_total_revenue/total_company_revenue)*100, 2) AS cumulative_revenue_pct
    FROM CumulativeRevenue
)

SELECT Product_id
    , Product_Name
    , Product_Family
    , ROUND(product_revenue, 2) AS product_revenue
    , ROUND(running_total_revenue, 2) AS running_total_revenue
    , cumulative_revenue_pct
    , CASE 
        WHEN cumulative_revenue_pct <= 80.00 THEN 'Top 80%'
        ELSE 'Remaining 20%'
    END AS pareto_classification
FROM ParetoCalculation
"""
insight6 = pd.read_sql(pareto_analysis, conn)
insight6

,Product_id,Product_Name,Product_Family,product_revenue,running_total_revenue,cumulative_revenue_pct,pareto_classification
0,2659,Veronica prostrata L.,Asteraceae,103373.05,103373.05,0.34,Top 80%
1,2381,Pleurothallis domingensis Cogn.,Cyperaceae,93576.82,196949.87,0.65,Top 80%
2,2561,Typha angustifolia L.,Fabaceae,92107.39,289057.26,0.96,Top 80%
3,2940,Verbena urticifolia L. var. leiocarpa L.M. Per...,Asteraceae,89288.14,378345.40,1.26,Top 80%
4,2352,Epilobium glaberrimum Barbey ssp. glaberrimum,Boraginaceae,88998.85,467344.25,1.55,Top 80%
...,...,...,...,...,...,...,...
910,2670,Triplasis purpurea (Walter) Chapm.,Hydrophyllaceae,5160.08,30055814.24,99.93,Remaining 20%
911,2945,Dietes grandiflora N.E. Br.,Rosaceae,5158.24,30060972.48,99.95,Remaining 20%
912,2310,Viola tomentosa M.S. Baker & J.C. Clausen,Menyanthaceae,5143.18,30066115.66,99.97,Remaining 20%
913,2357,Pityrogramma √ómackenneyi W.H. Wagner,Asteraceae,5134.50,30071250.16,99.98,Remaining 20%


### Top 5 Product Family contribution to Revenue

In [47]:
top5_product_family = """
SELECT ph.Product_Family
     , SUM(pf.Sales_USD) Revenue
FROM plant_fact AS pf
JOIN plant_hierarchy AS ph
ON pf.Product_id = ph.Product_Name_id
GROUP BY ph.Product_Family
ORDER BY Revenue DESC
LIMIT 5
"""
insight7 = pd.read_sql(top5_product_family, conn)
insight7

,Product_Family,Revenue
0,Asteraceae,3238198.45
1,Fabaceae,2365522.81
2,Poaceae,1711733.88
3,Rosaceae,1035645.79
4,Scrophulariaceae,1009802.59


### Top 5 Country contribution to Revenue

In [51]:
top5_country = """
WITH total_revenue AS(
    SELECT SUM(Sales_USD) AS sum_revenue
    FROM plant_fact
)

SELECT a.country_code
    , ROUND(SUM(pf.Sales_USD)/(SELECT sum_revenue FROM total_revenue)*100, 2) AS contribution_pct
FROM plant_fact AS pf
JOIN accounts AS a
ON pf.Account_id = a.Account_id
GROUP BY a.country_code
ORDER BY contribution_pct DESC
LIMIT 5
"""
insight8 = pd.read_sql(top5_country, conn)
insight8

,country_code,contribution_pct
0,CN,33.20
1,BR,7.51
2,PH,6.87
3,PT,5.48
4,PL,5.08
